# ESM-DB Flatfile Consolidation

**Authors:** Spina Cianetti  
**License:** [GPL-3.0](https://www.gnu.org/licenses/gpl-3.0.html)  
This code is released under the GNU General Public License v3.0.

---

This notebook consolidates the per-event spectral acceleration (SA) flatfile CSVs
produced by the ESM-DB download pipeline into a single unified DataFrame.

**Inputs:**

| Input | Description |
|---|---|
| `<output_dir>/*_SA.csv` | Per-event SA flatfile CSVs (semicolon-separated) |
| `events_dataframe.csv` | Previously consolidated event flatfile (for re-analysis) |

**Outputs:**

| Output file | Description |
|---|---|
| `events_dataframe.csv` | Consolidated flatfile sorted by event origin time |

**Sections:**
1. Setup and configuration
2. Read and consolidate per-event SA flatfiles

## 1. Setup and configuration

In [1]:
import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_DIR  = '/home/jovyan/shared/users/spina/ESM25/output/'
OUTPUT_CSV = 'events_dataframe.csv'
FIGURE_DIR = 'FIGURE'
os.makedirs(FIGURE_DIR, exist_ok=True)

# ── Magnitude columns in priority order ──────────────────────────────────────
# The first non-NaN value in this order is used as the preferred magnitude.
MAG_COLS = ['emec_mw', 'mw', 'ml', 'ms', 'mb', 'md', 'm']

# ── SEED code columns that must be read as strings ───────────────────────────
SEED_DTYPES = {
    'station_code': str,
    'network_code': str,
    'channel_code': str,
    'location_code': str,
}

## 2. Read and consolidate per-event SA flatfiles

Reads all `*_SA.csv` files from the input directory, skips empty files,
concatenates them into a single DataFrame and saves the result sorted by
event origin time.

In [2]:
all_files = sorted(glob.glob(os.path.join(INPUT_DIR, '*_SA.csv')))
print(f'SA CSV files found: {len(all_files)}')

df_list = []
n_skipped = 0

for csv_path in all_files:
    try:
        df = pd.read_csv(
            csv_path,
            sep=';',
            index_col=False,
            dtype=SEED_DTYPES,
        )
        if df.empty:
            print(f'[Skipped — empty] {os.path.basename(csv_path)}')
            n_skipped += 1
            continue
        df['source_file'] = os.path.basename(csv_path)  # track origin
        df_list.append(df)
    except Exception as e:
        print(f'[Error] {os.path.basename(csv_path)}: {e}')
        n_skipped += 1

if df_list:
    df_all = pd.concat(df_list, ignore_index=True)
    print(f'\nConsolidated {len(df_list)} files into {df_all.shape[0]:,} rows '
          f'and {df_all.shape[1]} columns.')
    print(f'Skipped / errored: {n_skipped}')
else:
    df_all = pd.DataFrame()
    print('No valid CSV files found.')

# Sort by event origin time and save
df_all_sorted = df_all.sort_values(by='event_time').reset_index(drop=True)
df_all_sorted.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved → {OUTPUT_CSV}')

SA CSV files found: 6998

Consolidated 6998 files into 101,668 rows and 357 columns.
Skipped / errored: 0

Saved → events_dataframe.csv


### Re-load the consolidated flatfile

The cell below re-reads `events_dataframe.csv` with the correct column dtypes.
Run this cell if you want to analyse an existing consolidated file without
re-running the concatenation above.

In [3]:
# Mixed-type column indices (from DtypeWarning at runtime):
# adjust if the column layout of events_dataframe.csv changes.
MIXED_COLS = {18: str, 22: str, 24: str, 26: str, 27: str,
              28: str, 33: str, 36: str, 38: str, 41: str, 58: str}

df_finale = pd.read_csv(OUTPUT_CSV, dtype=MIXED_COLS, low_memory=False)

print(f'Loaded: {df_finale.shape[0]:,} rows  {df_finale.shape[1]} columns')
print(f'Processing status counts:')
print(df_finale['processing_status'].value_counts(dropna=False).to_string())

Loaded: 101,668 rows  357 columns
Processing status counts:
processing_status
processed             65434
bad quality record    36234
